# Bandits: Learning While Deciding

**PSAM 3707 / UN 3707: Persuasion at Scale**
Columbia University, Spring 2026

This notebook accompanies Lecture 20 (April 13). We'll build three bandit algorithms from scratch, watch them learn in real time, and see why adaptive experimentation beats static A/B testing.

**Papers:**
- Offer-Westort, Coppock & Green (2021). "Adaptive Experimental Design." *AJPS*.
- Netflix Technology Blog (2017). "Artwork Personalization at Netflix."
- Salganik, Dodds & Watts (2006). "Experimental Study of Inequality and Unpredictability." *Science*.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Persuasion-at-Scale/bandits/blob/main/notebooks/bandits.ipynb)

In [ ]:
# Install dependencies (runs in Colab)
!pip install -q numpy matplotlib scipy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Reproducibility
np.random.seed(42)

# Clean plot style
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 13,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

## Part 1: The Multi-Armed Bandit Problem

Imagine you're a campaign manager with **5 different email subject lines** for a fundraising appeal. Each one has a different (unknown) success rate. You have 1,000 emails to send. How do you figure out which subject line works best while wasting as few emails as possible on bad ones?

This is the **multi-armed bandit problem**: you have K "arms" (options), each with an unknown payout rate. Every pull gives you two things:
1. A **reward** (did the person donate?)
2. **Information** (what's this arm's true rate?)

The tension between using what you know (**exploit**) and learning more (**explore**) is the core of the problem.

In [ ]:
# Set up the problem: 5 campaign email subject lines with different true success rates.
# The campaign manager doesn't know these rates. The bandit algorithm must discover them.

true_rates = [0.05, 0.08, 0.15, 0.10, 0.03]  # true donation rates per subject line
arm_names = [
    '"Urgent: Democracy at Stake"',
    '"Can you chip in $5?"',
    '"Your neighbor already donated"',   # <-- the best arm (social proof framing)
    '"Here\'s what we\'ll do with your $"',
    '"Last chance to make a difference"',
]
K = len(true_rates)  # number of arms
T = 1000             # total emails to send

print("The bandit problem:")
print(f"  {K} subject lines (arms)")
print(f"  {T} emails to send (pulls)")
print(f"  True rates are UNKNOWN to the algorithm")
print()
print("Subject lines:")
for i, (name, rate) in enumerate(zip(arm_names, true_rates)):
    print(f"  Arm {i}: {name}  (true rate: {rate:.0%}, hidden)")

## Part 2: Three Strategies

We'll implement three bandit algorithms and run them on the same problem. Each one balances explore/exploit differently:

1. **Epsilon-Greedy**: exploit 90% of the time, explore randomly 10%
2. **Upper Confidence Bound (UCB)**: pick the arm with the highest (estimate + uncertainty bonus)
3. **Thompson Sampling**: sample from each arm's belief distribution, pick the highest sample

In [ ]:
def pull_arm(arm_index):
    """Simulate pulling an arm. Returns 1 (success) or 0 (failure)."""
    return int(np.random.random() < true_rates[arm_index])


def run_epsilon_greedy(epsilon=0.1):
    """Epsilon-greedy: exploit (1-epsilon) of the time, explore epsilon of the time."""
    # Track successes and attempts for each arm
    successes = np.zeros(K)
    attempts = np.zeros(K)
    choices = []       # which arm was chosen each round
    rewards = []       # reward each round
    cum_reward = []    # cumulative reward over time

    for t in range(T):
        if np.random.random() < epsilon or t < K:
            # Explore: pick a random arm (or ensure each arm is tried once)
            arm = t if t < K else np.random.randint(K)
        else:
            # Exploit: pick the arm with the highest observed average
            rates = successes / np.maximum(attempts, 1)
            arm = np.argmax(rates)

        # Pull the arm and record the result
        reward = pull_arm(arm)
        successes[arm] += reward
        attempts[arm] += 1
        choices.append(arm)
        rewards.append(reward)
        cum_reward.append(sum(rewards))

    return choices, rewards, cum_reward


def run_ucb(c=2.0):
    """Upper Confidence Bound: pick the arm with the highest (mean + exploration bonus)."""
    successes = np.zeros(K)
    attempts = np.zeros(K)
    choices = []
    rewards = []
    cum_reward = []

    for t in range(T):
        if t < K:
            # Try each arm once
            arm = t
        else:
            # Compute UCB score for each arm
            rates = successes / attempts
            # Exploration bonus: sqrt(log(t) / n_i)
            bonus = c * np.sqrt(np.log(t) / attempts)
            arm = np.argmax(rates + bonus)

        reward = pull_arm(arm)
        successes[arm] += reward
        attempts[arm] += 1
        choices.append(arm)
        rewards.append(reward)
        cum_reward.append(sum(rewards))

    return choices, rewards, cum_reward


def run_thompson_sampling():
    """Thompson Sampling: sample from each arm's Beta posterior, pick the highest sample."""
    # Beta(alpha, beta) prior for each arm. Start with Beta(1,1) = uniform.
    alpha = np.ones(K)   # prior successes + 1
    beta_param = np.ones(K)    # prior failures + 1
    choices = []
    rewards = []
    cum_reward = []

    for t in range(T):
        # Sample from each arm's posterior
        samples = [np.random.beta(alpha[i], beta_param[i]) for i in range(K)]
        arm = np.argmax(samples)

        reward = pull_arm(arm)
        # Update the posterior: success -> alpha += 1, failure -> beta += 1
        alpha[arm] += reward
        beta_param[arm] += (1 - reward)
        choices.append(arm)
        rewards.append(reward)
        cum_reward.append(sum(rewards))

    return choices, rewards, cum_reward


print("Three strategies defined. Let's run them!")

In [ ]:
# Run all three strategies
np.random.seed(42)
eg_choices, eg_rewards, eg_cum = run_epsilon_greedy()

np.random.seed(42)
ucb_choices, ucb_rewards, ucb_cum = run_ucb()

np.random.seed(42)
ts_choices, ts_rewards, ts_cum = run_thompson_sampling()

# Also run a static A/B test (equal allocation) for comparison
np.random.seed(42)
ab_choices = [t % K for t in range(T)]  # round-robin across all arms
ab_rewards = [pull_arm(arm) for arm in ab_choices]
ab_cum = np.cumsum(ab_rewards).tolist()

print(f"Total donations collected:")
print(f"  A/B test (equal split):   {sum(ab_rewards):>3d} / {T}")
print(f"  Epsilon-greedy (e=0.1):   {sum(eg_rewards):>3d} / {T}")
print(f"  UCB (c=2):                {sum(ucb_rewards):>3d} / {T}")
print(f"  Thompson Sampling:        {sum(ts_rewards):>3d} / {T}")
print(f"  Oracle (always best arm): {sum(np.random.random(T) < max(true_rates)):>3d} / {T}")

### Figure 1: Cumulative Rewards Over Time

This is the key comparison. The y-axis shows total donations collected so far. The x-axis is the number of emails sent. Better algorithms pull ahead faster because they stop wasting emails on bad subject lines.

In [ ]:
# Figure 1: Cumulative reward comparison
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(ab_cum, label='A/B test (equal split)', color='gray', linewidth=2, alpha=0.7)
ax.plot(eg_cum, label='Epsilon-greedy', color='#e74c3c', linewidth=2)
ax.plot(ucb_cum, label='UCB', color='#3498db', linewidth=2)
ax.plot(ts_cum, label='Thompson Sampling', color='#2ecc71', linewidth=2)

# Oracle line
oracle_rate = max(true_rates)
ax.plot([0, T], [0, oracle_rate * T], '--', color='black', alpha=0.4, label=f'Oracle ({oracle_rate:.0%} rate)')

ax.set_xlabel('Emails Sent')
ax.set_ylabel('Total Donations Collected')
ax.set_title('Bandit Algorithms vs. Static A/B Testing')
ax.legend(loc='upper left', fontsize=11)
plt.tight_layout()
plt.show()

**What you should see:** All three bandit algorithms collect more donations than the static A/B test. Thompson Sampling tracks closest to the oracle (the theoretical best if you already knew the answer). The A/B test wastes resources by continuing to send bad subject lines throughout the experiment.

This is exactly what Offer-Westort, Coppock & Green (2021) showed: adaptive designs identify winners faster and waste fewer participants on losing treatments.

### Figure 2: How Each Algorithm Allocates Emails Over Time

The next figure shows *which arm* each algorithm chooses over time. A good algorithm should quickly concentrate on the best arm (Arm 2: "Your neighbor already donated").

In [ ]:
# Figure 2: Allocation over time (smoothed)
# For each strategy, compute the fraction of pulls going to each arm in a rolling window

def allocation_over_time(choices, window=50):
    """Compute smoothed allocation fractions over time."""
    fracs = np.zeros((T, K))
    for t in range(T):
        start = max(0, t - window)
        window_choices = choices[start:t+1]
        for k in range(K):
            fracs[t, k] = sum(1 for c in window_choices if c == k) / len(window_choices)
    return fracs

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
colors = ['#e74c3c', '#e67e22', '#2ecc71', '#3498db', '#9b59b6']
strategies = [
    ('Epsilon-Greedy', eg_choices),
    ('UCB', ucb_choices),
    ('Thompson Sampling', ts_choices),
]

for ax, (name, choices) in zip(axes, strategies):
    fracs = allocation_over_time(choices)
    # Stacked area chart
    ax.stackplot(range(T), fracs.T, labels=[f'Arm {i}' for i in range(K)],
                 colors=colors, alpha=0.8)
    ax.set_title(name, fontsize=14)
    ax.set_xlabel('Email #')
    ax.set_ylim(0, 1)
    # Mark the best arm
    ax.axhline(y=0.2, color='white', linestyle=':', alpha=0.3)

axes[0].set_ylabel('Fraction of Emails to Each Arm')
axes[2].legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize=10)
plt.suptitle('How Each Algorithm Shifts Traffic to the Best Arm Over Time', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**What you should see:** Thompson Sampling concentrates almost all traffic on the best arm (green, Arm 2) within the first 100-200 pulls. Epsilon-greedy concentrates too, but keeps wasting 10% on random exploration even at pull 1,000. UCB converges but may take longer because it gives generous exploration bonuses to under-sampled arms.

This is the Offer-Westort et al. result: Thompson Sampling naturally shifts 80%+ of traffic to the winner, while the static design (A/B) keeps allocating equally throughout.

## Part 3: Thompson Sampling Under the Hood

Let's watch the Beta posteriors evolve as Thompson Sampling learns. This is the Bayesian heart of the algorithm: each arm has a probability distribution representing our *belief* about its true rate. As we observe successes and failures, the distribution tightens around the truth.

In [ ]:
# Figure 3: Beta posterior evolution for Thompson Sampling
# Show snapshots at t = 10, 50, 100, 500

np.random.seed(42)
alpha = np.ones(K)
beta_param = np.ones(K)
snapshots = {10: None, 50: None, 100: None, 500: None}

for t in range(T):
    samples = [np.random.beta(alpha[i], beta_param[i]) for i in range(K)]
    arm = np.argmax(samples)
    reward = pull_arm(arm)
    alpha[arm] += reward
    beta_param[arm] += (1 - reward)
    if (t + 1) in snapshots:
        snapshots[t + 1] = (alpha.copy(), beta_param.copy())

fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
x = np.linspace(0, 0.35, 300)
colors = ['#e74c3c', '#e67e22', '#2ecc71', '#3498db', '#9b59b6']

for ax, (t_snap, (a, b)) in zip(axes, snapshots.items()):
    for i in range(K):
        y = stats.beta.pdf(x, a[i], b[i])
        ax.plot(x, y, color=colors[i], linewidth=2, label=f'Arm {i}')
        # Mark the true rate with a vertical line
        ax.axvline(true_rates[i], color=colors[i], linestyle=':', alpha=0.4)
    ax.set_title(f'After {t_snap} pulls', fontsize=13)
    ax.set_xlabel('Donation Rate')

axes[0].set_ylabel('Probability Density')
axes[3].legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize=10)
plt.suptitle("Thompson Sampling: Beliefs Converge to the Truth", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**What you should see:** At t=10, all arms have wide, overlapping distributions (we know very little). By t=50, the best arm (green, Arm 2) starts separating. By t=500, the green distribution is tall and narrow, centered on the true rate of 15%. The worst arms have barely been sampled because Thompson Sampling stopped pulling them early.

The dotted vertical lines show the true rates. Notice how the posterior peaks converge to these true values as evidence accumulates. Arms that were pulled less have wider posteriors (more uncertainty), because the algorithm chose not to waste pulls on them.

## Part 4: Netflix Thumbnail Simulation (Contextual Bandits)

Netflix doesn't show the same thumbnail to everyone. Different users see different images for the same show. This is a **contextual bandit**: the best arm depends on who the user is.

Let's simulate this. We have 3 user types (action fans, drama fans, comedy fans) and 3 thumbnails for a show. The click rate depends on the match between user type and thumbnail.

In [ ]:
# Contextual bandit: Netflix thumbnail personalization
# True click rates: rows = user types, columns = thumbnails
# Each user type has a different best thumbnail

click_rates = np.array([
    # Action thumbnail | Drama thumbnail | Comedy thumbnail
    [0.20,               0.05,             0.08],   # Action fans
    [0.06,               0.22,             0.07],   # Drama fans
    [0.04,               0.03,             0.18],   # Comedy fans
])

user_types = ['Action fans', 'Drama fans', 'Comedy fans']
thumbnails = ['Action shot', 'Drama scene', 'Comedy moment']

# Thompson Sampling per user type (separate Beta posteriors for each user-thumbnail pair)
n_users = 3
n_thumbs = 3
alpha_ctx = np.ones((n_users, n_thumbs))
beta_ctx = np.ones((n_users, n_thumbs))

T_ctx = 3000  # total user visits
np.random.seed(42)

# Track choices and cumulative clicks
ctx_choices = []
ctx_clicks = []
# Also track what a "no personalization" baseline would do (always show the same thumbnail)
baseline_clicks = []

for t in range(T_ctx):
    # A random user visits
    user = np.random.randint(n_users)

    # Thompson Sampling: sample from posteriors for this user type
    samples = [np.random.beta(alpha_ctx[user, j], beta_ctx[user, j]) for j in range(n_thumbs)]
    thumb = np.argmax(samples)

    # Did they click?
    clicked = int(np.random.random() < click_rates[user, thumb])
    alpha_ctx[user, thumb] += clicked
    beta_ctx[user, thumb] += (1 - clicked)

    ctx_choices.append((user, thumb))
    ctx_clicks.append(clicked)

    # Baseline: always show the first thumbnail (action shot) to everyone
    baseline_clicked = int(np.random.random() < click_rates[user, 0])
    baseline_clicks.append(baseline_clicked)

print(f"Results after {T_ctx} user visits:")
print(f"  Contextual bandit (personalized): {sum(ctx_clicks)} clicks ({sum(ctx_clicks)/T_ctx:.1%})")
print(f"  Always show 'Action shot':        {sum(baseline_clicks)} clicks ({sum(baseline_clicks)/T_ctx:.1%})")
print(f"  Improvement from personalization:  {(sum(ctx_clicks) - sum(baseline_clicks)) / sum(baseline_clicks):.0%}")

In [ ]:
# Figure 4: What the contextual bandit learned
# Show the learned allocation (which thumbnail is shown to which user type)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors_thumb = ['#e74c3c', '#2980b9', '#f39c12']

for u in range(n_users):
    ax = axes[u]
    # Count how many times each thumbnail was shown to this user type
    counts = np.zeros(n_thumbs)
    for (user, thumb) in ctx_choices:
        if user == u:
            counts[thumb] += 1
    total = counts.sum()
    fracs = counts / total

    bars = ax.bar(thumbnails, fracs, color=colors_thumb, alpha=0.8, edgecolor='white')
    ax.set_title(user_types[u], fontsize=13)
    ax.set_ylim(0, 1)
    ax.set_ylabel('Fraction Shown' if u == 0 else '')

    # Annotate with true click rates
    for j, bar in enumerate(bars):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
                f'true: {click_rates[u,j]:.0%}', ha='center', fontsize=9, color='gray')

plt.suptitle('Contextual Bandit Learns to Match Thumbnails to User Types', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**What you should see:** The algorithm learned to show action fans the action thumbnail, drama fans the drama thumbnail, and comedy fans the comedy thumbnail. Each user type gets a personalized experience, and the overall click rate is much higher than showing everyone the same image.

This is what Netflix does at 20M+ requests per second. The same show looks different depending on who you are. The bandit learned which match works best by experimenting on real users.

## Part 5: Salganik's MusicLab - Path Dependence

Salganik, Dodds & Watts (2006) showed that social influence creates unpredictable outcomes. The same song can become a hit in one "world" and a flop in another, depending on early random events. This matters for bandits: if early signals are noisy, an adaptive algorithm might lock in on the wrong winner.

Let's simulate 8 parallel "worlds" like Salganik's MusicLab.

In [ ]:
# Salganik's MusicLab: 48 songs, 8 parallel worlds
# In the "independent" condition, popularity depends only on quality.
# In the "social influence" condition, downloads compound: visible download counts
# create a feedback loop where early winners keep winning.

n_songs = 48
n_worlds = 8
n_listeners = 2000  # per world

# True "quality" of each song (fixed across worlds)
np.random.seed(123)
true_quality = np.random.beta(2, 5, n_songs)  # skewed: most songs are mediocre
true_quality.sort()  # sort by quality for clarity

def run_musiclab_world(social_influence=True):
    """Simulate one world of the MusicLab experiment."""
    downloads = np.zeros(n_songs)

    for listener in range(n_listeners):
        if social_influence:
            # Probability of downloading is a mix of quality and social signal
            # Social signal = fraction of total downloads (rich-get-richer)
            if downloads.sum() == 0:
                social_signal = np.ones(n_songs) / n_songs
            else:
                social_signal = downloads / downloads.sum()
            # Weight: 30% quality, 70% social signal
            prob = 0.3 * true_quality + 0.7 * social_signal
            prob = prob / prob.sum()  # normalize
        else:
            # Independent condition: probability proportional to quality only
            prob = true_quality / true_quality.sum()

        # Each listener downloads one song
        chosen = np.random.choice(n_songs, p=prob)
        downloads[chosen] += 1

    return downloads

# Run 8 worlds with social influence and 1 independent world
np.random.seed(42)
independent_world = run_musiclab_world(social_influence=False)
social_worlds = [run_musiclab_world(social_influence=True) for _ in range(n_worlds)]

print(f"Simulated {n_worlds} social-influence worlds + 1 independent world")
print(f"  {n_songs} songs, {n_listeners} listeners per world")

In [ ]:
# Figure 5: Salganik replication -- quality vs. market share across 8 worlds

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Independent condition (quality predicts success)
ax = axes[0]
market_share = independent_world / independent_world.sum()
ax.scatter(true_quality, market_share, s=40, alpha=0.7, color='#2c3e50')
# Fit a trend line
z = np.polyfit(true_quality, market_share, 1)
x_trend = np.linspace(true_quality.min(), true_quality.max(), 100)
ax.plot(x_trend, np.polyval(z, x_trend), '--', color='gray', alpha=0.7)
ax.set_xlabel('Song Quality (true, fixed)')
ax.set_ylabel('Market Share')
ax.set_title('Independent Condition\n(No social information)', fontsize=13)

# Right: Social influence (8 worlds, same songs, wildly different outcomes)
ax = axes[1]
world_colors = plt.cm.Set2(np.linspace(0, 1, n_worlds))
for w in range(n_worlds):
    market_share_w = social_worlds[w] / social_worlds[w].sum()
    ax.scatter(true_quality, market_share_w, s=20, alpha=0.5, color=world_colors[w],
               label=f'World {w+1}')
ax.set_xlabel('Song Quality (true, fixed)')
ax.set_ylabel('Market Share')
ax.set_title('Social Influence Condition\n(8 parallel worlds)', fontsize=13)

plt.suptitle("Salganik's MusicLab: Social Influence Creates Unpredictability",
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**What you should see:** In the independent condition (left), quality is a reasonable predictor of market share: better songs get more downloads. In the social influence condition (right), outcomes are wildly scattered. The same song can be popular in one world and ignored in another. Early random events compound through the feedback loop.

**Why this matters for bandits:** An adaptive algorithm that learns from early signals has an advantage, but in a social system, those early signals can be self-fulfilling prophecies. A message that happens to work on the first 10 people might look like a winner, and the algorithm will send more people to it, making it work on the *next* 10 because of social proof. The algorithm can lock in on a winner that is only a winner *because* the algorithm chose it.

## Part 6: The Regret Perspective

"Regret" measures the cost of not knowing the best arm from the start. At each pull, the regret is: (best arm's true rate) minus (chosen arm's true rate). Cumulative regret tells you how much total reward you lost due to exploration.

A good bandit algorithm has regret that grows slowly (logarithmically). A bad one (or a static A/B test) has regret that grows linearly forever.

In [ ]:
# Figure 6: Cumulative regret comparison
best_rate = max(true_rates)

def cumulative_regret(choices):
    """Compute cumulative regret: sum of (best_rate - chosen_arm_rate) over time."""
    regret = [best_rate - true_rates[c] for c in choices]
    return np.cumsum(regret)

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(cumulative_regret(ab_choices), label='A/B test', color='gray', linewidth=2, alpha=0.7)
ax.plot(cumulative_regret(eg_choices), label='Epsilon-greedy', color='#e74c3c', linewidth=2)
ax.plot(cumulative_regret(ucb_choices), label='UCB', color='#3498db', linewidth=2)
ax.plot(cumulative_regret(ts_choices), label='Thompson Sampling', color='#2ecc71', linewidth=2)

ax.set_xlabel('Emails Sent')
ax.set_ylabel('Cumulative Regret (donations lost)')
ax.set_title('Regret: How Many Donations Did Exploration Cost?')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

**What you should see:** The A/B test's regret grows linearly (a straight line going up). It never stops wasting pulls on bad arms. The bandit algorithms' regret curves flatten out: after finding the best arm, they stop losing donations. Thompson Sampling flattens fastest.

The gap between the A/B test line and the Thompson Sampling line at t=1,000 is the number of donations you'd have lost by running a standard experiment instead of an adaptive one. This is the practical argument for bandits in Offer-Westort et al.

## Summary: What We Learned

| Strategy | How it explores | Strength | Weakness |
|----------|----------------|----------|----------|
| A/B test | Equal allocation forever | Simple, clean inference | Wastes resources on losers |
| Epsilon-greedy | Random, fixed 10% | Very simple | Never stops exploring |
| UCB | Uncertainty bonus | Principled, guaranteed bounds | Can over-explore |
| Thompson Sampling | Sample from beliefs | Adapts naturally, fast convergence | Needs a prior |
| Contextual bandit | Per-user Thompson Sampling | Personalizes | More complex |

**The key insight from the lecture:** bandits learn while earning. Static A/B tests separate "learning" and "doing." Bandits combine them. This is what makes Netflix thumbnails, campaign messaging, and (in HW4) your own strategies possible.

**Connection to HW4:** In the bandit tournament, you write a natural-language strategy that Claude translates to Python. Your strategy competes against your classmates'. Think about: when should you explore? When should you exploit? How do you handle uncertainty?